# Framingham Heart Study: Exploratory Data Analysis & Preprocessing Design
This notebook conducts systematic EDA on the Framingham dataset to prepare for a production 10-year Coronary Heart Disease (CHD) risk predictor.

### Objectives:
1. Examine class balance, missing values, duplicates, and feature distributions.
2. Formulate clinical rationales for feature exclusion (e.g., dropping non-clinical education).
3. Analyze correlation and collinearity (e.g. sysBP and diaBP, currentSmoker and cigsPerDay).
4. Design a strict, zero-leakage preprocessing pipeline.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/framingham.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

### 1. Data Types & Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct.round(2)})
missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

### 2. Clinical Decision: Dropping education
- education represents the highest educational attainment (1-4).
- **Clinical rationale:** Educational attainment is a socioeconomic proxy rather than a direct pathophysiological determinant in cardiovascular screening calculators (such as Framingham 10-year risk or ACC/AHA ASCVD risk estimators). Using socioeconomic factors in biological risk scores introduces demographic bias across distinct healthcare systems.
- Therefore, education is dropped from all downstream modeling.

In [ ]:
df_clean = df.drop(columns=['education'])
print('Remaining features:', df_clean.columns.tolist())

### 3. Target Distribution (TenYearCHD)

In [ ]:
target_counts = df_clean['TenYearCHD'].value_counts()
print(f'Negative (0): {target_counts[0]} ({target_counts[0]/len(df_clean):.1%})')
print(f'Positive (1): {target_counts[1]} ({target_counts[1]/len(df_clean):.1%})')
print(f'Imbalance Ratio: ~1:{round(target_counts[0]/target_counts[1], 1)}')

### 4. Correlation Analysis

In [ ]:
corr = df_clean.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Correlation Matrix of Clinical Features')
plt.show()

### 5. Key Clinical Insights & Preprocessing Strategy
1. **Target Imbalance:** Positive class rate is ~15.2%. Standard accuracy is misleading; optimization and threshold tuning will focus on ROC-AUC, PR-AUC, and F2-score to prioritize recall in clinical screening.
2. **Missingness:** glucose has 388 missing rows (~9.15%), 	otChol (50), BPMeds (53), cigsPerDay (29), BMI (19), heartRate (1).
3. **Imputation:** Median imputation + missing indicators within scikit-learn Pipeline so imputation parameters are fit strictly on training folds without data leakage.
4. **Blood Pressure Collinearity:** sysBP and diaBP have high correlation (r = 0.78). Systolic blood pressure is clinically the strongest predictor of ASCVD in older adults, while diastolic pressure provides additional context.
5. **Smoking Features:** currentSmoker correlates directly with cigsPerDay. Median imputation will preserve consistency.
